# Transform Drivers  Data

> 1. Read bronze `drivers` table
>  1. Keep only the columns required for analytics (Drop `url` column)
>  1. Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfbirth` → `date_of_birth`)
>  1. Concatenate `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform the value to Title Case
>  1. Remove duplicate records
>  1. Transform values of column `nationality` to Title Case
>  1. Write the transformed data to silver `drivers` table

#### Entity Relationship Diagram - Formula1 Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")



In [0]:
%run "../00-common/01.environment-config"


In [0]:
val bronze_table = catalog_name + "." + bronze_schema + "." + "drivers "
val silver_table = catalog_name + "." + silver_schema + "." + "drivers"

#### Step 1 - Read bronze `drivers` table

In [0]:
val drivers_df= spark.table(bronze_table)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
val drivers_selected_df = drivers_df.drop("url")

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (driverId → driver_id, dateOfbirth → date_of_birth)



In [0]:
val drivers_renamed_df = drivers_selected_df
.withColumnRenamed("driverId","driver_id")
.withColumnRenamed("dateOfBirth", "date_of_birth")

#### Step 4 - Concatenate name.givenName and name.familyName to create a new column called driver_name  and transform the value to Title Case



In [0]:
import org.apache.spark.sql.functions.{col,initcap,concat_ws}
val drivers_concatenated_df=drivers_renamed_df
.withColumn("driver_name",initcap(concat_ws(" ",col("name.givenName"),col("name.familyName")))).drop("name")

In [0]:
display(drivers_concatenated_df)

#### Step 5 - Remove duplicate records

In [0]:
val drivers_distinct_df = drivers_concatenated_df.dropDuplicates("driver_id")

In [0]:
display(drivers_distinct_df)

#### Step 7 - Transform values of columns `nationality`  to Title Case


In [0]:
import org.apache.spark.sql.functions.{initcap,col}
val drivers_final_df= drivers_distinct_df
.withColumn("nationality",initcap(col("nationality")))

In [0]:
display(drivers_final_df)

#### Step 8 - Write the transformed data to silver `constructors` table

In [0]:
drivers_final_df.write.format("delta").mode("overwrite")
        .saveAsTable(silver_table)


In [0]:
display(spark.table(silver_table))